# 07 深层 Transformer 为什么需要残差缩放和初始化策略？

## 面试回答主线

深层残差网络的问题不是只有梯度消失，也包括每层新增分支的方差累积。一个常见思路是让残差分支初始化或输出尺度随深度衰减，例如乘以 $1/\sqrt{L}$，使总残差能量在深度增加时仍可控。它不是万能开关：缩放过强会减慢特征学习，缩放过弱会使激活和更新不稳定。面试时应同时说明初始化、残差系数、norm、学习率和深度是耦合配方。实验用 24 层手写残差 MLP 记录每层 RMS，比较未缩放与 $1/\sqrt{L}$ 缩放，并故意把残差系数设大制造失败。

**核心公式：** 若每层残差近似独立且方差为 $\sigma^2$，直接相加后方差可随 $L$ 增长；令 $x_{l+1}=x_l+\alpha F_l(x_l)$ 且 $\alpha=1/\sqrt L$，可把累计尺度控制在常数量级。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
class DeepResidual(nn.Module):  # 手写深层残差 MLP 以观察逐层尺度。
    def __init__(self, depth, alpha):  # 接收深度和残差缩放系数。
        super().__init__()  # 初始化模块父类。
        self.alpha = alpha  # 保存残差系数。
        self.weights = nn.ParameterList([nn.Parameter(torch.randn(3, 3) / math.sqrt(3.0)) for _ in range(depth)])  # 创建每层投影矩阵。
    def forward(self, value):  # 返回输出和逐层 RMS 轨迹。
        rms_trace = []  # 保存每层激活尺度。
        for weight in self.weights:  # 遍历每层残差矩阵。
            value = value + self.alpha * torch.tanh(value @ weight)  # 执行带缩放的残差更新。
            rms_trace.append(float(value.pow(2).mean().sqrt()))  # 记录当前层 RMS。
        return value, rms_trace  # 返回最终表示与尺度轨迹。
unscaled_model = DeepResidual(24, 1.0)  # 创建不缩放的基线网络。
_, unscaled_trace = unscaled_model(features)  # 运行基线前向传播。
baseline_metric = unscaled_trace[-1]  # 记录最后一层 RMS。
print(f'未缩放：第 1/12/24 层 RMS={unscaled_trace[0]:.3f}/{unscaled_trace[11]:.3f}/{unscaled_trace[-1]:.3f}')  # 展示方差累积。


未缩放：第 1/12/24 层 RMS=0.858/2.702/3.852


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
alpha = 1.0 / math.sqrt(24.0)  # 按深度设置残差缩放系数。
scaled_model = DeepResidual(24, alpha)  # 创建缩放后的同深度网络。
scaled_output, scaled_trace = scaled_model(features)  # 运行核心前向传播。
core_metric = scaled_trace[-1]  # 保存缩放后最后一层 RMS。
print(f'缩放系数={alpha:.4f}，第 1/12/24 层 RMS={scaled_trace[0]:.3f}/{scaled_trace[11]:.3f}/{scaled_trace[-1]:.3f}')  # 输出逐层对照。


缩放系数=0.2041，第 1/12/24 层 RMS=0.709/0.863/0.875


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=3.852297
核心机制     | 指标=0.875015


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **残差缩放** 的关键状态与更新路径。生产模型要把缩放策略固化到配置和 checkpoint 兼容逻辑；深度、层类型和并行切分变化时必须重新验证，不应复用旧阈值。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
unsafe_model = DeepResidual(24, 2.4)  # 故意把残差系数放大到危险水平。
_, unsafe_trace = unsafe_model(features)  # 运行失败配置。
failure_metric = unsafe_trace[-1]  # 记录失败配置末层 RMS。
fix_metric = scaled_trace[-1]  # 复用正确缩放的末层 RMS。
print(f'失败：过大残差系数末层 RMS={failure_metric:.3f}；修复：1/sqrt(L) 末层 RMS={fix_metric:.3f}')  # 展示残差尺度门限的价值。


失败：过大残差系数末层 RMS=16.507；修复：1/sqrt(L) 末层 RMS=0.875


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产模型要把缩放策略固化到配置和 checkpoint 兼容逻辑；深度、层类型和并行切分变化时必须重新验证，不应复用旧阈值。

**常见坑：** 只缩放初始化而忽略残差输出，或把每层都缩放到几乎没有学习信号，都会得到看似稳定但欠训练的网络。

**延伸追问：** DeepNorm、ReZero、LayerScale 与 $1/\sqrt L$ 的假设差异是什么？MoE 层的 token 路由不均衡会如何改变残差尺度？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert len(unscaled_trace) == 24  # 验证基线记录了每一层的激活。
assert len(scaled_trace) == 24  # 验证缩放网络记录了每一层的激活。
assert torch.isfinite(scaled_output).all()  # 验证缩放输出保持有限。
assert failure_metric > fix_metric  # 验证过大的残差系数扩大了末层尺度。
